# SLAP-GRO — Phase 2: Warehouse Layout & Visualization

This notebook provides interactive 2D and 3D visual analysis of the warehouse layer, comparing:
1. **Mode A (`SyntheticWarehouse`)**: Paper replication setup with 847 units across 3 blocks (A, B, C) and 25 aisles.
2. **Mode B (`CSVWarehouse`)**: Real industrial layout with 2,292 locations and 44 navigation support points.

In [ ]:
import sys
import os
from pathlib import Path

# Find project root by searching upwards for pyproject.toml
p = Path(os.getcwd()).resolve()
while p != p.parent:
    if (p / 'pyproject.toml').exists():
        break
    p = p.parent
project_root = p

src_dir = project_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from slap_gro.warehouse import build_warehouse, Location, NavigationPoint

# Build both layouts
wh_paper = build_warehouse('paper')
wh_real = build_warehouse('real')

print(f"Mode A (Paper): {wh_paper.total_locations} locations, {wh_paper.total_capacity:,} total slots")
print(f"Mode B (Real):  {wh_real.total_locations} locations, {wh_real.total_capacity:,} total slots")

## 1. Summary Comparison Table

In [ ]:
summary_csv = project_root / 'outputs' / 'tables' / 'warehouse_summary.csv'
if not summary_csv.exists():
    from scripts.build_warehouse import main as build_wh_main
    build_wh_main()

df_summary = pd.read_csv(summary_csv)
df_summary

## 2. Real Warehouse Layout (`CSVWarehouse` — 2,292 Locations)
Visualizing 2D layout floor-by-floor with support navigation points (LC, CC, RC) and I/O depot.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.flatten()

nav_pts = wh_real.navigation_points.values()
nav_x = [pt.x for pt in nav_pts]
nav_y = [pt.y for pt in nav_pts]
depot = wh_real.get_depot()

for z_lvl in [1, 2, 3, 4]:
    ax = axes[z_lvl - 1]
    locs_lvl = wh_real.get_locations_by_level(z_lvl)
    xs = [loc.x for loc in locs_lvl]
    ys = [loc.y for loc in locs_lvl]

    # Plot storage locations
    ax.scatter(xs, ys, c='steelblue', s=12, alpha=0.6, label=f'Locations ({len(locs_lvl)})')

    # Plot navigation points
    if z_lvl == 1:
        ax.scatter(nav_x, nav_y, c='orange', marker='^', s=40, edgecolors='black', label=f'Support Points ({len(nav_pts)})')
        ax.scatter([depot.x], [depot.y], c='red', marker='*', s=150, edgecolors='black', label='Depot (0, 0)')

    ax.set_title(f'Real Warehouse — Floor Z={z_lvl} ({len(locs_lvl)} units)', fontsize=12, fontweight='bold')
    ax.set_xlabel('X (meters)')
    ax.set_ylabel('Y (meters)')
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.legend(loc='upper right', fontsize=9)

plt.tight_layout()
plt.show()

## 3. Synthetic Warehouse Layout (`SyntheticWarehouse` — 847 Units)
Visualizing the paper's 3-block layout (A: 7 aisles, B: 9 aisles, C: 9 aisles) across 4 levels.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

colors = {'A': '#1f77b4', 'B': '#2ca02c', 'C': '#ff7f0e'}
for blk in ['A', 'B', 'C']:
    locs_blk = wh_paper.get_locations_by_block(blk)
    xs = [loc.x for loc in locs_blk]
    ys = [loc.y for loc in locs_blk]
    ax.scatter(xs, ys, color=colors[blk], s=15, alpha=0.7, label=f'Block {blk} ({len(locs_blk)} units)')

# Depot
depot_paper = wh_paper.get_depot()
ax.scatter([depot_paper.x], [depot_paper.y], color='red', marker='*', s=200, edgecolors='black', label='Depot')

# Navigation points
nav_paper = wh_paper.navigation_points.values()
ax.scatter([p.x for p in nav_paper], [p.y for p in nav_paper], color='purple', marker='x', s=50, label=f'Waypoints ({len(nav_paper)})')

ax.set_title('Synthetic 3D Warehouse (2D Plane Projection) — Blocks A, B, C', fontsize=14, fontweight='bold')
ax.set_xlabel('X (meters)')
ax.set_ylabel('Y (meters)')
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()